# Model training

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("..")) # for src imports

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
    print("   CUDA version:", torch.version.cuda)

In [ ]:
NUM_CLASSES = 19
DATASET_DIR = "../data/dataset" # train / val / test
MODEL_DIR = "../models/best_unet.pt" # model save

# SPECS1 =  CPU 10700KF, RAM 32GB, GPU RTX 2080S 8GB VRAM
#Original = (1024, 2048) # 2,097,152
#IMG_SIZE = (512, 1024)  # 524,288
IMG_SIZE = (256, 512)   # 131,072 # OK using SPECS1 (8 batch, 12 workers) 20 epoch = ~23 mins (~6GB VRAM used)
#IMG_SIZE = (128, 256)   # 32,768 # EASY using SPECS1 (8 batch, 12 workers) 20 epoch = ~10 mins (~2GB VRAM used)
#IMG_SIZE = (64, 128)    # 8,192

BATCH_SIZE = 8
NUM_WORKERS = min(12, os.cpu_count() or 1)
print("Dataloader workers:", NUM_WORKERS)

Hyperparameters

In [ ]:
# u-net model
BASE = 32 # 
# optimizer
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
# training
EPOCHS = 20


Transforms

In [ ]:
from torchvision import transforms
from src.utils import mask_to_long_tensor

# get transforms for images and masks
image_transform = transforms.Compose([
    # BILINEAR averages neighboring pixels to produce smooth gradients
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor() # tensor scaled to [0, 1]
])

label_transform = transforms.Compose([
    # NEAREST preserves original values
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(), # tensor without scaling (stays 0-255)
    # remove dummy channel dimension and cast to int64 because PyTorch's CrossEntropyLoss requires this for segmentation. [1, H, W] -> [H, W]
    transforms.Lambda(mask_to_long_tensor) # lambda function here does not work in notebook so moved to utils.py
])

Dataloaders

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.dataset import CityscapesDataset


train_ds = CityscapesDataset("../data/dataset/train", image_transform, label_transform)
valid_ds = CityscapesDataset("../data/dataset/val", image_transform, label_transform)
test_ds = CityscapesDataset("../data/dataset/test", image_transform, label_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True, # useful for stochastic gradient descent
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)


Check formats

In [ ]:
from src.model import UNet
model = UNet().to(device)

# grab one bactch to inspect shape
# [B = Batch Size, 3 = RGB, H = Height, W = Width]
imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # should be [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # should be [B,H,W] int64

# run model with no gradient (reduces memory usage) to inspect output shape
# [B = Batch Size, 19 = Classes, H = Height, W = Width]
with torch.no_grad():
    out = model(imgs.to(next(model.parameters()).device))
print("logits:", out.shape)  # should be [B,19,H,W]

Create model

In [ ]:
model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device)

Train model

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(ignore_index=255) # ignore unlabelled
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks in tqdm(loader, leave=False):
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).long()

            logits = model(imgs)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
    return total_loss / len(loader)

best_val = float("inf")

for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, True)
    va = run_epoch(valid_loader, False)
    print(f"epoch {epoch:02d}  train {tr:.4f}  val {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), MODEL_DIR)

Load saved model

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR), map_location=device, weights_only=True)
model.eval()

Show sample output

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from cityscapesscripts.helpers.labels import labels

# Official colours and names for trainIds
# Source: https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py

# keep only trainIds (0-18), sort by trainId
classes = sorted(
    [l for l in labels if 0 <= l.trainId < 19],
    key=lambda l: l.trainId
)

# matplotlib expects float RGB values [0, 1], so divide by 255 to convert
colors = np.array([l.color for l in classes]) / 255
# create colour map with 19 colours + black for unlabelled
cmap = ListedColormap(colors)
cmap.set_over("black")

# get a test image and prediction
imgs, masks = next(iter(test_loader))
model.eval()
with torch.no_grad():
    pred = model(imgs.to(device)).argmax(1).cpu()

# images
img = imgs[0].permute(1, 2, 0).cpu()
true_mask = masks[0].cpu()
pred_mask = pred[0]

# create plot
fig, ax = plt.subplots(2, 1, figsize=(8, 8))

# matplotlib expects (height, width, 3) instead of (3, height, width) so rearrange
ax[0].imshow(img)
ax[0].imshow(true_mask, cmap=cmap, vmin=0, vmax=18, alpha=0.5)
ax[0].set_title("Image with true mask")

ax[1].imshow(pred_mask, cmap=cmap, vmin=0, vmax=18)
ax[1].set_title("Predicted Mask")

# hide plots axis, numbers, borders
for a in ax:
    a.axis("off")

# colour and name + add black for unlabelled
handles = [Patch(color=colors[i], label=l.name) for i, l in enumerate(classes)]
handles.append(Patch(color="black", label="unlabelled"))

fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.95, 0.5), title="Classes")

plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

get Pixel Accuracy, Mean IoU (mIoU), Mean Class Accuracy

In [ ]:
import torch
import numpy as np
from cityscapesscripts.helpers.labels import labels

# Get the official Cityscapes trainId names
classes = sorted(
    [l for l in labels if 0 <= l.trainId < NUM_CLASSES],
    key=lambda l: l.trainId
)
class_names = [l.name for l in classes]

# Confusion matrix: rows = true class, columns = predicted class
confusion_matrix = torch.zeros(
    NUM_CLASSES, NUM_CLASSES, dtype=torch.int64
)

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs = imgs.to(device)
        masks = masks.to(device).long()

        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        # Ignore invalid labels, if any
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        # Encode each true/predicted pair as a single index
        encoded = (
            NUM_CLASSES * masks[valid] + preds[valid]
        )

        confusion_matrix += torch.bincount(
            encoded.cpu(),
            minlength=NUM_CLASSES**2
        ).reshape(NUM_CLASSES, NUM_CLASSES)

# Calculate metrics
cm = confusion_matrix.numpy().astype(np.float64)

true_positives = np.diag(cm)
ground_truth_count = cm.sum(axis=1)
predicted_count = cm.sum(axis=0)

# Per-class IoU
union = ground_truth_count + predicted_count - true_positives
iou = np.divide(
    true_positives,
    union,
    out=np.full(NUM_CLASSES, np.nan),
    where=union != 0
)

# Per-class accuracy
class_accuracy = np.divide(
    true_positives,
    ground_truth_count,
    out=np.full(NUM_CLASSES, np.nan),
    where=ground_truth_count != 0
)

# Overall pixel accuracy
pixel_accuracy = (
    true_positives.sum() / cm.sum()
    if cm.sum() > 0 else float("nan")
)

# Mean metrics, excluding classes absent from both prediction
# and ground truth
mean_iou = np.nanmean(iou)
mean_class_accuracy = np.nanmean(class_accuracy)

print(f"Pixel Accuracy:       {pixel_accuracy:.4%}")
print(f"Mean IoU (mIoU):      {mean_iou:.4%}")
print(f"Mean Class Accuracy:  {mean_class_accuracy:.4%}")

print("\nPer-class results:")
print("-" * 48)

for name, class_iou, acc in zip(
    class_names, iou, class_accuracy
):
    iou_str = f"{class_iou:.4%}" if np.isfinite(class_iou) else "N/A"
    acc_str = f"{acc:.4%}" if np.isfinite(acc) else "N/A"
    print(f"{name:20s} IoU: {iou_str:>10s} | Accuracy: {acc_str:>10s}")